## Install Required Dependecies

In [2]:
!pip install langchain_openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.8/84.8 kB 4.3 MB/s eta 0:00:00


In [3]:
!pip install -qU langchain-community faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


## Necessary Imports

In [4]:
# Necessary Imports

import os
import csv
import pandas as pd
from openai import OpenAI
from google.colab import userdata

from enum import Enum
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate


from abc import ABC, abstractmethod
from typing import Any, List, Tuple, Iterator, Dict

from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings


from concurrent.futures import ThreadPoolExecutor, as_completed
from langchain_core.documents import Document

from tqdm import tqdm

In [5]:
# Load OPENAI API KEY

api_key = userdata.get('OPENAI_API_KEY')
os.environ["OPENAI_API_KEY"] = api_key

In [6]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

## Intent Classification Layer

In [7]:
class IntentCategory(str, Enum):
    FACTUAL_FAQ = "FACTUAL_FAQ"
    PROCEDURAL_GUIDANCE = "PROCEDURAL_GUIDANCE"
    COMMUNITY_EXPERIENCE = "COMMUNITY_EXPERIENCE"
    EVIDENCE_BASED = "EVIDENCE_BASED"
    COMPLEX_MULTI_HOP = "COMPLEX_MULTI_HOP"


class IntentOutput(BaseModel):
    intent: IntentCategory = Field(
        description="The single most appropriate intent category for the user query"
    )


INTENT_CLASSIFIER_SYSTEM_PROMPT = """
You are an intent classification engine for a COVID-19 question-answering system.

Your task:
Given a user query, classify it into EXACTLY ONE intent category.

You must always return one category, even if the query does not clearly match any rules.
Use semantic understanding, not just keywords.

--------------------
INTENT CATEGORIES
--------------------

FACTUAL_FAQ:
- Short, direct factual questions
- Definitions, symptoms, transmission, vaccines, timelines
- Example: "What are the symptoms of COVID-19?"

PROCEDURAL_GUIDANCE:
- Questions asking for steps, actions, prevention, or advice
- Often includes "how", "what should I do", "can I", but do NOT rely only on keywords
- Example: "How can I protect myself from COVID-19?"

COMMUNITY_EXPERIENCE:
- Informal, opinionated, rumor-based, or social media influenced queries
- Mentions of people, beliefs, claims, or uncertainty
- Example: "People say masks don't work, is that true?"

EVIDENCE_BASED:
- Research-oriented, scientific, or evidence-seeking questions
- Mentions studies, data, research, clinical findings, or comparisons
- Example: "What does research say about COVID survival on surfaces?"

COMPLEX_MULTI_HOP:
- Long, multi-part, ambiguous, or mixed-intent questions
- Requires synthesising multiple facts or evidence
- Use this ONLY when no single intent clearly dominates
- Example: "How does COVID affect elderly people and what precautions should families take?"

--------------------
DECISION RULES (GUIDANCE, NOT HARD RULES)
--------------------

1. Use semantic meaning first, keywords second.
2. If multiple intents seem present:
   - Choose the MOST dominant intent.
3. If the query is long, compound, or unclear:
   - Prefer COMPLEX_MULTI_HOP.
4. If safety or scientific grounding is implied:
   - Prefer EVIDENCE_BASED over FACTUAL_FAQ.
5. Never return multiple categories.
6. Never invent new categories.
7. Never explain your reasoning.

Return ONLY the intent category in structured form.
"""

intent_llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

intent_classifier = intent_llm.with_structured_output(IntentOutput)

def classify_intent_llm(query: str) -> IntentCategory:
    result = intent_classifier.invoke(
        [
            {"role": "system", "content": INTENT_CLASSIFIER_SYSTEM_PROMPT},
            {"role": "user", "content": query}
        ]
    )
    return result.intent.value

## Query Expansion Layer

In [8]:
class QueryExpander:
    """
    LLM-based query expansion for multi-query retrieval.
    """

    DELIMITER = "#next-question#"

    def __init__(
        self,
        model_name: str = "gpt-4o-mini",
        n_variations: int = 3,
        temperature: float = 0.3,
    ):
        self.n_variations = n_variations
        self.llm = ChatOpenAI(
            model=model_name,
            temperature=temperature,
        )

        self.prompt = PromptTemplate(
            input_variables=["query", "n", "delimiter"],
            template="""
You are an expert search query rewriter for retrieval-augmented generation systems.

Rewrite the following user query into {n} alternative versions.
Each version should reflect a different perspective or phrasing
while preserving the original intent.

Separate each rewritten query using the token:
{delimiter}

User query:
"{query}"
""",
        )

    def expand(self, query: str) -> List[str]:
        formatted_prompt = self.prompt.format(
            query=query,
            n=self.n_variations,
            delimiter=self.DELIMITER,
        )

        response = self.llm.invoke(formatted_prompt).content

        expanded_queries = [
            q.strip()
            for q in response.split(self.DELIMITER)
            if q.strip()
        ]

        # Always include original query
        return [query] + expanded_queries


## Multi FAISS Retriever Function

In [9]:
class MultiFAISSRetriever:
    """
    Concurrent retriever supporting:
    - multiple FAISS indexes
    - multiple expanded queries
    - optional similarity scores
    """

    def __init__(
        self,
        vectorstores: List[FAISS],
        top_k: int = 5,
        max_workers: int = 8,
        use_scores: bool = False,
    ):
        self.vectorstores = vectorstores
        self.top_k = top_k
        self.max_workers = max_workers
        self.use_scores = use_scores

    def _search(
        self, vs: FAISS, query: str
    ) -> List[Tuple[Document, float]] | List[Document]:
        """
        Internal helper to perform either:
        - similarity_search
        - similarity_search_with_score
        """
        if self.use_scores:
            return vs.similarity_search_with_score(query, k=self.top_k)
        return vs.similarity_search(query, k=self.top_k)

    def retrieve(
        self, queries: List[str]
    ) -> List[Document] | List[Tuple[Document, float]]:
        """
        Executes concurrent retrieval across all queries and indexes.
        Deduplicates documents across sources.
        """

        seen_doc_ids = set()
        results = []

        with ThreadPoolExecutor(max_workers=self.max_workers) as executor:
            futures = [
                executor.submit(self._search, vs, q)
                for vs in self.vectorstores
                for q in queries
            ]

            for future in as_completed(futures):
                retrieved = future.result()

                for item in retrieved:
                    if self.use_scores:
                        doc, score = item
                    else:
                        doc = item
                        score = None

                    # Robust deduplication key
                    doc_id = (
                        doc.metadata.get("id")
                        if doc.metadata and "id" in doc.metadata
                        else hash(doc.page_content)
                    )

                    if doc_id in seen_doc_ids:
                        continue

                    seen_doc_ids.add(doc_id)

                    if self.use_scores:
                        results.append((doc, score))
                    else:
                        results.append(doc)

        return results


## Context Retrieval Layer

In [10]:
# Load Multiple FAISS Vector Stores from Google Drive

def load_faiss_vectorstores(
    vectorstore_paths: List[str],
    embedding_model: OpenAIEmbeddings,
) -> List[FAISS]:
    """
    Loads multiple FAISS vector stores from disk.
    """
    vectorstores = []

    for path in vectorstore_paths:
        vs = FAISS.load_local(
            folder_path=path,
            embeddings=embedding_model,
            allow_dangerous_deserialization=True,  # required for FAISS
        )
        vectorstores.append(vs)

    return vectorstores

# Vector Stores Path

vectorstore_paths = {
  "source_1": "/content/drive/MyDrive/MS-LJMU/Vector-Store/Store-1-3072-Kaggle-COVID-19-FAQ",

  "source_2": "/content/drive/MyDrive/MS-LJMU/Vector-Store/Store-2-3072-COVID-QA-community",

  "source_3": "/content/drive/MyDrive/MS-LJMU/Vector-Store/Store-3-3072-COUGH-FAQ-ENG",

  "source_4": "/content/drive/MyDrive/MS-LJMU/Vector-Store/Store-4-3072-COVID-QA-MASTER"

}

# Intent Routing

INTENT_ROUTING = {
    "FACTUAL_FAQ": ["source_1", "source_3"],
    "PROCEDURAL_GUIDANCE": ["source_1", "source_3"],
    "COMMUNITY_EXPERIENCE": ["source_2"],
    "EVIDENCE_BASED": ["source_4"],
    "COMPLEX_MULTI_HOP": ["source_1", "source_3", "source_4"],
}

In [11]:
user_query = "What are the common symptoms of COVID-19?"

In [12]:
# Indentify the intent and load respective vector stores
intent = classify_intent_llm(user_query)

sources = INTENT_ROUTING[intent]
stores_paths = [vectorstore_paths[source] for source in sources]

indexes = load_faiss_vectorstores(
    vectorstore_paths=stores_paths,
    embedding_model=embeddings,
)


# # Expand query
expander = QueryExpander(n_variations=3)
expanded_queries = expander.expand(user_query)

In [13]:
# Retrieve documents
retriever = MultiFAISSRetriever(
    vectorstores=indexes,
    top_k=5,
    use_scores=True
)

retrieved_docs_with_scores = retriever.retrieve(expanded_queries)

## Post-Retrieval Layer

### Reranking (Cross-Encoders)

The most prominent post-retrieval technique discussed is reranking, which addresses the limitations of standard vector searches.

- The Problem: Initial searches (bi-encoders) rely on distance-based similarity between query and document embeddings, which may occasionally retrieve documents that are semantically close but contextually unaligned with the user's true intent.

- The Solution: A cross-encoder model is used to score each retrieved chunk against the original user query. Unlike bi-encoders, cross-encoders can identify more complex and nuanced relationships between the question and the text.

- The Workflow: The system retrieves a broader pool of candidates—typically N×K chunks if query expansion was used—and then the reranker reorders them, allowing the system to keep only the absolute best top K results for the final prompt.

In [16]:
from sentence_transformers import CrossEncoder

In [17]:
class CrossEncoderReranker:
    """
    Cross-encoder reranker for post-retrieval optimisation.
    """

    def __init__(
        self,
        model_name: str = "cross-encoder/ms-marco-MiniLM-L-6-v2",
        top_k: int = 5,
        batch_size: int = 16,
    ):
        self.model = CrossEncoder(model_name)
        self.top_k = top_k
        self.batch_size = batch_size

    def rerank(
        self,
        query: str,
        documents: List[Document],
    ) -> List[Tuple[Document, float]]:
        """
        Reranks retrieved documents using cross-encoder scores.
        """

        pairs = [(query, doc.page_content) for doc in documents]

        scores = self.model.predict(
            pairs,
            batch_size=self.batch_size,
        )

        reranked = list(zip(documents, scores))
        reranked.sort(key=lambda x: x[1], reverse=True)

        return reranked[: self.top_k]


### Prompt Compression (Context Refinement Layer)

Why Prompt Compression?

Even after reranking:
- Chunks may be verbose
- Redundant details increase cost
- Long prompts → attention bias + hallucination

Goal:
- Preserve meaning, remove noise.

In [18]:
class PromptCompressor:
    """
    Compresses retrieved documents while preserving factual content.
    """

    def __init__(
        self,
        model_name: str = "gpt-4o-mini",
        temperature: float = 0.0,
        max_tokens: int = 256,
    ):
        self.llm = ChatOpenAI(
            model=model_name,
            temperature=temperature,
        )

        self.prompt = PromptTemplate(
            input_variables=["query", "context"],
            template="""
You are a system that compresses retrieved context for a question-answering system.

Given the user query and the retrieved text:
- Remove irrelevant or redundant information
- Preserve all medically important facts
- Do NOT add new information
- Keep the output concise and factual

User query:
{query}

Retrieved context:
{context}

Compressed context:
""",
        )

        self.max_tokens = max_tokens

    def compress(
        self,
        query: str,
        documents: List[Document],
    ) -> List[str]:
        compressed_chunks = []

        for doc in documents:
            formatted_prompt = self.prompt.format(
                query=query,
                context=doc.page_content,
            )

            response = self.llm.invoke(
                formatted_prompt,
                max_tokens=self.max_tokens,
            )

            compressed_chunks.append(response.content.strip())

        return compressed_chunks


### Post Retrieval Optimisation Pipeline

In [19]:
def post_retrieval_optimisation(
    query: str,
    retrieved_docs_with_scores: List[Tuple[Document, float]],
    rerank_top_k: int = 5,
) -> List[str]:
    """
    Post-retrieval optimisation:
    1. Cross-encoder reranking
    2. Prompt compression
    """

    # Strip FAISS scores
    retrieved_docs = [doc for doc, _ in retrieved_docs_with_scores]

    # Reranking
    reranker = CrossEncoderReranker(top_k=rerank_top_k)
    reranked_docs_with_scores = reranker.rerank(
        query=query,
        documents=retrieved_docs,
    )

    reranked_docs = [doc for doc, _ in reranked_docs_with_scores]

    # Prompt Compression
    compressor = PromptCompressor()
    compressed_context = compressor.compress(
        query=query,
        documents=reranked_docs,
    )

    return compressed_context


In [20]:
compressed_context = post_retrieval_optimisation(
    query=user_query,
    retrieved_docs_with_scores=retrieved_docs_with_scores,
    rerank_top_k=5
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

## Generation Layer

### Prompt Engineering

In [26]:
SYSTEM_PROMPT = """
You are a helpful, factual, and cautious AI assistant.
Your task is to answer user questions using ONLY the provided context.
If the answer cannot be found in the context, clearly say:
"I do not have enough information to answer this question."

Do not make assumptions.
Do not add external knowledge.
Be concise and accurate.
"""


INSTRUCTION_PROMPT = """
Context:
{context}

User Question:
{question}

Instructions:
- Use the context above as your primary source of information
- Do not hallucinate or fabricate facts
- If the context is insufficient, explicitly say so

Answer:
"""


def build_augmented_prompt(
    question: str,
    context_chunks: List[str],
) -> List[dict]:
    """
    Builds a chat-style prompt with system + user messages.
    """

    context_text = "\n\n".join(
        f"[Source {i+1}]\n{chunk}"
        for i, chunk in enumerate(context_chunks)
    )

    user_prompt = INSTRUCTION_PROMPT.format(
        context=context_text,
        question=question,
    )

    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]


### Inference Service

In [21]:
class GenerationOutput(BaseModel):
    """
    Structured output schema for final RAG generation.
    """
    llm_response: str = Field(
        description="Final grounded answer generated using the provided context."
    )

In [24]:
class StructuredInferenceService:
    """
    Final RAG inference service with enforced structured output.
    """

    def __init__(
        self,
        model_name: str = "gpt-4o-mini",
        temperature: float = 0,
        max_tokens: int = 1012,
    ):
        self.llm = ChatOpenAI(
            model=model_name,
            temperature=temperature,
            max_tokens=max_tokens,
        ).with_structured_output(GenerationOutput)

    def generate(self, messages: List[Dict]) -> GenerationOutput:
        """
        Generates a structured response.
        """
        return self.llm.invoke(messages)

In [27]:
inference_llm = StructuredInferenceService()

result = inference_llm.generate(
    messages=build_augmented_prompt(
        user_query,
        compressed_context,
    )
)

final_answer = result.llm_response

In [32]:
final_answer

'The common symptoms of COVID-19 include fever, cough, and tiredness. Other symptoms may include aches and pains, nasal congestion, runny nose, sore throat, chills, muscle or body aches, headache, new loss of taste or smell, abdominal pain/discomfort, nausea, vomiting, and diarrhea.'

### Pre to Post-Retrieval to Inference Pipelines

In [ ]:
def pre_to_postretrieval_pipeline(user_query: str):

  # Indentify the intent and load respective vector stores
  intent = classify_intent_llm(user_query)
  sources = INTENT_ROUTING[intent]
  stores_paths = [vectorstore_paths[source] for source in sources]

  indexes = load_faiss_vectorstores(
      vectorstore_paths=stores_paths,
      embedding_model=embeddings,
  )

  # Query Expansion
  expander = QueryExpander(n_variations=3)
  expanded_queries = expander.expand(user_query)

  # Retrieve documents (Multi Index, Multi Query Retrieval)
  retriever = MultiFAISSRetriever(
      vectorstores=indexes,
      top_k=5,
      use_scores=True
  )

  retrieved_docs_with_scores = retriever.retrieve(expanded_queries)

  # Post Retrieval Optimization Pipeline
  compressed_context = post_retrieval_optimisation(
      query=user_query,
      retrieved_docs_with_scores=retrieved_docs_with_scores,
      rerank_top_k=5
  )

  return compressed_context


def single_inference_pipeline(user_query: str, context: List[str]):

  inference_llm = StructuredInferenceService()

  result = inference_llm.generate(
      messages=build_augmented_prompt(
          user_query,
          context,
      )
  )

  final_answer = result.llm_response

  return final_answer

### Batch Inference Service

In [36]:
def run_batch_inference_structured(
    inference_llm: StructuredInferenceService,
    queries: List[str],
    contexts: List[List[str]],
    output_csv_path: str,
) -> None:
    """
    Runs batch RAG inference with structured output and saves to CSV.
    """

    assert len(queries) == len(contexts), (
        "Queries and contexts must have the same length."
    )

    with open(output_csv_path, mode="w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=[
                "query_id",
                "user_query",
                "context_used",
                "llm_response",
            ],
        )
        writer.writeheader()

        for idx, (query, context_chunks) in enumerate(
            tqdm(zip(queries, contexts), total=len(queries))
        ):
            messages = build_augmented_prompt(
                question=query,
                context_chunks=context_chunks,
            )

            structured_output = inference_llm.generate(messages)

            writer.writerow(
                {
                    "query_id": idx,
                    "user_query": query,
                    "context_used": " || ".join(context_chunks),
                    "llm_response": structured_output.llm_response,
                }
            )


In [37]:
user_queries = [
    "What are the symptoms of COVID-19?",
    "How does COVID-19 spread?",
]

In [38]:
contexts = []
for user_query in user_queries:
  compressed_context = pre_to_postretrieval_pipeline(user_query)
  contexts.append(compressed_context)

In [44]:
inference_llm = StructuredInferenceService()

run_batch_inference_structured(
    inference_llm=inference_llm,
    queries=user_queries,
    contexts=contexts,
    output_csv_path="/content/drive/MyDrive/MS-LJMU/Batch-Output/test_260126_batch_inference_structured.csv",
)

100%|██████████| 2/2 [00:04<00:00,  2.20s/it]
